In [21]:
from train_patchtst import load_and_process_data
import argparse
import torch
import torch.nn.functional as F
from PatchTST import Model as PatchTST
import numpy as np
import pickle as pkl
import os
import base64
import io
from PIL import Image
from openai import OpenAI

In [ ]:

# 1. Create the configuration object with all necessary arguments
# These should match the arguments used during training.
args = argparse.Namespace(
    # Data loading args
    seq_len=24,
    pred_len=1,
    batch_size=1,
    seed=10,

    # Model architecture args
    task_name='classification',
    d_model=64,
    n_heads=16,
    e_layers=2,
    d_ff=256,
    dropout=0.1,
    activation='relu',
    enc_in=5,
    num_class=2,
    factor=1,

    # Other args (not directly used for inference but good to have)
    learning_rate=0.001,
    num_epochs=100,
    patience=10,
    checkpoint_dir='expl_results/weather_ny'
)

# 2. Load the test data
_, _, test_loader = load_and_process_data(args)
print("Test loader created successfully.")

# 3. Instantiate the model and load the checkpoint
patchtst_model = PatchTST(args)
checkpoint_path = 'expl_results/weather_ny/patchtst_checkpoint.pth'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patchtst_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
patchtst_model.to(device)
patchtst_model.eval() # Set the model to evaluation mode

print(f"Model loaded from {checkpoint_path}")

# 4. Get a single batch and perform inference
with torch.no_grad():
    # Get one batch from the test loader
    batch_x, batch_y = next(iter(test_loader))
    batch_x = batch_x.float().to(device)

    # Get the raw model output (logits)
    logits = patchtst_model(batch_x, None, None, None)

    # Convert logits to probabilities
    probabilities = torch.softmax(logits, dim=1)

    # Get the final prediction
    _, prediction = torch.max(probabilities, 1)

# 5. Display the results
print("\n--- Inference Results for a Single Sample ---")
print(f"Input Shape: {batch_x.shape}")
print(f"True Label: {batch_y.item()}")
print(f"Logits: {logits.cpu().numpy()}")
print(f"Probabilities: {probabilities.cpu().numpy()}")
print(f"Final Prediction: {prediction.item()}")

In [32]:
# %%
# New cell for KL Divergence analysis

# 1. Load the saved logits and probabilities
checkpoint_dir = 'expl_results/weather_ny'
train_logits = np.load(os.path.join(checkpoint_dir, 'train_logits.npy'))
val_logits = np.load(os.path.join(checkpoint_dir, 'val_logits.npy'))
train_probs = np.load(os.path.join(checkpoint_dir, 'train_probs.npy'))
val_probs = np.load(os.path.join(checkpoint_dir, 'val_probs.npy'))

# Concatenate train and val sets
all_logits = np.concatenate((train_logits, val_logits), axis=0)
all_probs = np.concatenate((train_probs, val_probs), axis=0)

print(f"Loaded and combined train/val data. Logits shape: {all_logits.shape}, Probs shape: {all_probs.shape}")

# 2. Recreate the data context to get labels and original indices
data_path = './dataset/weather_ny/'
with open(os.path.join(data_path, 'indices.pkl'), 'rb') as f:
    original_indices = pkl.load(f)
with open(os.path.join(data_path, 'rain.pkl'), 'rb') as f:
    rain_labels = pkl.load(f)

data_size = len(original_indices)

num_train = int(data_size * 0.6)
num_test = int(data_size * 0.2)
num_vali = data_size - num_train - num_test

seq_len_day = 1
pred_len = 1

train_idx = np.arange(num_train - seq_len_day)
val_idx = np.arange(num_train - seq_len_day, num_train + num_vali - seq_len_day)
test_idx = np.arange(num_train + num_vali - seq_len_day, num_train + num_vali + num_test - seq_len_day)

all_indices_map = np.concatenate((train_idx, val_idx), axis=0)

all_original_indices = [original_indices[i] for i in all_indices_map]
all_labels = [int(rain_labels[i + pred_len]) for i in all_indices_map]

Loaded and combined train/val data. Logits shape: (1507, 2), Probs shape: (1507, 2)


In [11]:

# 3. Calculate KL Divergence from LOGITS
# Define a temperature for softening the distributions. T > 1 softens.
T = 2.0

# Get the test sample's logits from the previous cell
test_logits_tensor = logits.cpu()

kl_divergences = []
for i in range(all_logits.shape[0]):
    # Convert numpy array to tensor
    train_val_logits_tensor = torch.from_numpy(all_logits[i])

    # To calculate KL(P || Q), P is the target and the input is log(Q)
    # Let P be the test distribution and Q be the train/val distribution
    p_test_soft = F.softmax(test_logits_tensor / T, dim=-1)
    log_q_train_val_soft = F.log_softmax(train_val_logits_tensor / T, dim=-1)

    # Calculate the KL divergence
    kl_div = F.kl_div(log_q_train_val_soft.unsqueeze(0), p_test_soft, reduction='batchmean')
    kl_divergences.append(kl_div.item())

# 4. Bind KL divergence with labels, original indices, and class 0 probability
bound_data = list(zip(kl_divergences, all_labels, all_original_indices, all_probs[:, 0]))

# 5. Separate by class and find the top 3 with the smallest KL divergence
class_0_samples = sorted([item for item in bound_data if item[1] == 0], key=lambda x: x[0])
class_1_samples = sorted([item for item in bound_data if item[1] == 1], key=lambda x: x[0])

print("\n--- Top 3 Most Similar Samples (by KL Divergence on Logits) ---")

print("\nFor Class 0 (Not Rain):")
for kl, label, orig_idx, prob_0 in class_0_samples[:3]:
    print(f"  Original Index: {orig_idx}, KL Divergence: {kl:.6f}, Label: {label}, Prob(Not Rain): {prob_0:.4f}")

print("\nFor Class 1 (Rain):")
for kl, label, orig_idx, prob_0 in class_1_samples[:3]:
    print(f"  Original Index: {orig_idx}, KL Divergence: {kl:.6f}, Label: {label}, Prob(Not Rain): {prob_0:.4f}")




--- Top 3 Most Similar Samples (by KL Divergence on Logits) ---

For Class 0 (Not Rain):
  Original Index: 28056, KL Divergence: -0.000000, Label: 0, Prob(Not Rain): 0.8536
  Original Index: 1728, KL Divergence: 0.000000, Label: 0, Prob(Not Rain): 0.8539
  Original Index: 23448, KL Divergence: 0.000000, Label: 0, Prob(Not Rain): 0.8530

For Class 1 (Rain):
  Original Index: 35952, KL Divergence: 0.000001, Label: 1, Prob(Not Rain): 0.8544
  Original Index: 28296, KL Divergence: 0.000004, Label: 1, Prob(Not Rain): 0.8550
  Original Index: 27240, KL Divergence: 0.000011, Label: 1, Prob(Not Rain): 0.8509


In [16]:
image_base_path = './dataset/weather_ny/images/'
text_base_path = './dataset/weather_ny/txt/'

print("\n--- Retrieving Data for Most Similar Samples ---")

# Retrieve for Class 0
print("\n--- Class 0 (Not Rain) Similar Samples ---")
for i, (kl, label, orig_idx, prob_0) in enumerate(class_0_samples[:3]):
    image_path = os.path.join(image_base_path, f's_{orig_idx}.png')
    text_path = os.path.join(text_base_path, f's_{orig_idx}.txt')

    print(f"\nSample #{i+1} (Original Index: {orig_idx})")
    print(f"  - KL Divergence: {kl:.6f}")
    print(f"  - Image Path: {image_path}")

    try:
        with open(text_path, 'r') as f:
            text_content = f.read()
        print(f"  - Text Content: {text_content.strip()}")
    except FileNotFoundError:
        print(f"  - Text Content: Not Found at {text_path}")

# Retrieve for Class 1
print("\n--- Class 1 (Rain) Similar Samples ---")
for i, (kl, label, orig_idx, prob_0) in enumerate(class_1_samples[:3]):
    image_path = os.path.join(image_base_path, f's_{orig_idx}.png')
    text_path = os.path.join(text_base_path, f's_{orig_idx}.txt')

    print(f"\nSample #{i+1} (Original Index: {orig_idx})")
    print(f"  - KL Divergence: {kl:.6f}")
    print(f"  - Image Path: {image_path}")

    try:
        with open(text_path, 'r') as f:
            text_content = f.read()
        print(f"  - Text Content: {text_content.strip()}")
    except FileNotFoundError:
        print(f"  - Text Content: Not Found at {text_path}")



--- Retrieving Data for Most Similar Samples ---

--- Class 0 (Not Rain) Similar Samples ---

Sample #1 (Original Index: 28056)
  - KL Divergence: -0.000000
  - Image Path: ./dataset/weather_ny/images/s_28056.png
  - Text Content: Not Found at ./dataset/weather_ny/txt/s_28056.txt

Sample #2 (Original Index: 1728)
  - KL Divergence: 0.000000
  - Image Path: ./dataset/weather_ny/images/s_1728.png
  - Text Content: Over the past 24 hours, New York City has experienced stable atmospheric pressure, suggesting no immediate large-scale storm systems are approaching. Temperature remained relatively constant, indicating a steady air mass with minimal diurnal variation, likely due to overcast or humid conditions. Humidity levels were consistently elevated, which may contribute to a muggy feel and could support fog or light precipitation if conditions shift. Wind speed showed intermittent spikes but generally stayed low, implying calm surface conditions with little mixing of air layers. Wind dir

In [18]:
# Helper function to encode images
def encode_image_to_base64(image_path):
    """Reads an image file and returns its base64 encoded string."""
    with Image.open(image_path) as img:
        # Ensure image is in a web-friendly format like JPEG
        img = img.convert("RGB")
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [26]:
city = 'weather_ny'

city_full_name = {
    'weather_ny': 'New York City'
}

In [38]:
# --- 1. Gather all the necessary data ---

# Define paths
image_base_path = './dataset/weather_ny/images/'
text_base_path = './dataset/weather_ny/txt/'

# Get the original index of the test sample we are analyzing
# (It's the first sample from the test set)
test_sample_orig_idx = original_indices[test_idx[0]]
test_image_path = os.path.join(image_base_path, f's_{test_sample_orig_idx}.png')
test_text_path = os.path.join(text_base_path, f's_{test_sample_orig_idx}.txt')

print(test_text_path)

# --- 2. Construct the multi-modal prompt ---

# This will be a list of text and image dictionaries
content_parts_system = []

# Add the main instruction
content_parts_system.append({
    "type": "text",
    "text": "You are an expert weather forecaster. Your task is to predict whether it will rain in the next 24 hours based on a primary time-series chart and a summary of the weather from the past 24 hours."
})

content_parts = []
content_parts.append({
    "type": "text",
    "text": f"Your task is to predict whether it will rain in the next 24 hours in {city_full_name[city]}. To help you, I will provide several historical examples that a separate analysis model has identified as being highly influential. Analyze all the provided information and make a final prediction with a brief explanation."
})

# Add the primary test text summary
try:
    with open(test_text_path, 'r') as f:
        test_text_content = f.read().strip()
except FileNotFoundError:
    test_text_content = "Primary text summary not found."

print(test_text_content)

content_parts.append({"type": "text", "text": f"\n--- Primary Text Summary to Analyze ---\n{test_text_content}"})

# Add the primary test image to analyze
content_parts.append({"type": "text", "text": "\n--- Primary Chart to Analyze ---"})
content_parts.append({
    "type": "image_url",
    "image_url": f"data:image/jpeg;base64,{encode_image_to_base64(test_image_path)}"
})

# Add the influential "Not Rain" examples
content_parts.append({"type": "text", "text": "\n--- Influential Historical Examples (True Label: Not Rain) ---"})
for i, (kl, label, orig_idx, prob_0) in enumerate(class_0_samples[:3]):
    image_path = os.path.join(image_base_path, f's_{orig_idx}.png')
    text_path = os.path.join(text_base_path, f's_{orig_idx}.txt')

    try:
        with open(text_path, 'r') as f:
            text_content = f.read().strip()
    except FileNotFoundError:
        text_content = "Text summary not found."

    content_parts.append({"type": "text", "text": f"\nExample {i+1} (Not Rain): {text_content}"})
    content_parts.append({
        "type": "image_url",
        "image_url": f"data:image/jpeg;base64,{encode_image_to_base64(image_path)}"
    })

# Add the influential "Rain" examples
content_parts.append({"type": "text", "text": "\n--- Influential Historical Examples (True Label: Rain) ---"})
for i, (kl, label, orig_idx, prob_0) in enumerate(class_1_samples[:3]):
    image_path = os.path.join(image_base_path, f's_{orig_idx}.png')
    text_path = os.path.join(text_base_path, f's_{orig_idx}.txt')

    try:
        with open(text_path, 'r') as f:
            text_content = f.read().strip()
    except FileNotFoundError:
        text_content = "Text summary not found."

    content_parts.append({"type": "text", "text": f"\nExample {i+1} (Rain): {text_content}"})
    content_parts.append({
        "type": "image_url",
        "image_url": f"data:image/jpeg;base64,{encode_image_to_base64(image_path)}"
    })

# Add the final instruction
content_parts.append({
    "type": "text",
    "text": "\nBased on the primary chart, weather summary and all the influential historical examples, will it rain? Please format your response as follows: First, provide your final prediction by writing either `<Rain>` or `<Not Rain>`. Second, provide a brief justification for your decision."
})

36168
./dataset/weather_ny/txt/s_36168.txt
Primary text summary not found.


In [37]:

# --- 3. Call the Client ---

# Re-initialize the client
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY", "",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

messages = [
    {
        "role": "system",
        "content": content_parts_system
    },
    {
        "role": "user",
        "content": content_parts
    }
]

# Make the API call
response = client.chat.completions.create(
    model='qwen3-vl-plus',
    messages=messages,
    max_tokens=2048
)

# --- 4. Print the Final Explanation ---
print("\n--- Final Explanation from Vision-Language Model ---")
print(response.choices[0].message.content)



--- Final Explanation from Vision-Language Model ---
<Not Rain>

The primary chart shows stable air pressure (green line), constant temperature (blue line), and moderate, relatively steady humidity (red line) — all indicators of a stable air mass without the dynamic changes typically associated with rain. Wind speed (yellow line) is low and fluctuates mildly, while wind direction (black line) remains nearly flat, suggesting no frontal passage or significant moisture advection. These patterns closely resemble the “Not Rain” historical examples, particularly Example 2 and 3, which also featured stable pressure, steady temperature, and calm winds despite elevated or moderate humidity. No sharp drops in pressure, surges in humidity, or erratic wind shifts — common precursors to rain — are present. Therefore, conditions favor dry weather in the next 24 hours.
